# insurance-monitoring

**Model monitoring for UK insurance pricing: PSI, A/E ratios, Gini drift, and Murphy decomposition in one report.**

This notebook shows the core workflow in under two minutes: generate a synthetic motor portfolio, introduce three realistic failure modes, and run `MonitoringReport` to detect each one with a traffic-light verdict.

The problem this solves: aggregate A/E ratio reads 1.00 while the model is 15% cheap on under-25s and 15% expensive on mature drivers. Errors cancel at portfolio level. `insurance-monitoring` checks the features, not just the headline number.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/insurance-monitoring/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip install -q insurance-monitoring polars

## Synthetic data: reference period (training window)

We build a reference portfolio of 8,000 policies representing the model's training window. Predictions are from a Poisson model; actuals are drawn from that same process (no drift).

In [ ]:
import numpy as np
import polars as pl

rng = np.random.default_rng(2024)

# Reference period: 8,000 policies, well-calibrated model
n_ref = 8_000
pred_ref = rng.uniform(0.04, 0.18, n_ref)
act_ref  = rng.poisson(pred_ref).astype(float)

feat_ref = pl.DataFrame({
    "driver_age":  rng.integers(18, 75, n_ref).tolist(),
    "vehicle_age": rng.integers(0, 12, n_ref).tolist(),
    "ncd_years":   rng.integers(0, 6, n_ref).tolist(),
})

print(f"Reference period - policies: {n_ref:,}  |  A/E: {act_ref.sum()/pred_ref.sum():.4f}")
feat_ref.head()

## Synthetic data: monitoring period with induced drift

The monitoring period is 18 months after deployment. We introduce three failure modes that a simple A/E check would miss:

1. **Covariate shift** - young drivers (18-30) enter the book at double their reference rate
2. **Calibration drift** - model systematically underpredicts for the monitoring period (A/E - 1.08)
3. **Discrimination decay** - 25% of predictions are partially randomised, degrading Gini

In [ ]:
n_cur = 3_000
pred_cur_raw = rng.uniform(0.04, 0.18, n_cur)

# Failure mode 3: partial prediction randomisation (Gini decay)
noise_mask = rng.random(n_cur) < 0.25
pred_cur = np.where(noise_mask, rng.uniform(0.04, 0.18, n_cur), pred_cur_raw)

# Failure mode 2: calibration drift - actuals 8% above predictions
act_cur = rng.poisson(pred_cur * 1.08).astype(float)

# Failure mode 1: covariate shift - young drivers oversampled
driver_age_cur = np.concatenate([
    rng.integers(18, 30, int(n_cur * 0.40)),   # 40% young drivers vs ~15% in reference
    rng.integers(30, 75, n_cur - int(n_cur * 0.40)),
])
rng.shuffle(driver_age_cur)

feat_cur = pl.DataFrame({
    "driver_age":  driver_age_cur.tolist(),
    "vehicle_age": rng.integers(0, 12, n_cur).tolist(),
    "ncd_years":   rng.integers(0, 6, n_cur).tolist(),
})

print(f"Monitoring period - policies: {n_cur:,}  |  Raw A/E: {act_cur.sum()/pred_cur.sum():.4f}")
print(f"Under-25 share - reference: {((feat_ref['driver_age'] < 30).sum() / n_ref):.1%}  |  current: {((feat_cur['driver_age'] < 30).sum() / n_cur):.1%}")

## Run MonitoringReport

`MonitoringReport` runs all three checks in one call: PSI/CSI for feature distribution shifts, A/E ratio with Poisson confidence intervals, Gini drift z-test (arXiv 2510.04556), and Murphy decomposition (when `murphy_distribution` is set) to distinguish RECALIBRATE from REFIT.

In [ ]:
from insurance_monitoring import MonitoringReport

report = MonitoringReport(
    reference_actual=act_ref,
    reference_predicted=pred_ref,
    current_actual=act_cur,
    current_predicted=pred_cur,
    feature_df_reference=feat_ref,
    feature_df_current=feat_cur,
    features=["driver_age", "vehicle_age", "ncd_years"],
    murphy_distribution="poisson",
)

print(f"Recommendation: {report.recommendation}")
print()
print(report.to_polars())

## Dig into what the report found

The Murphy decomposition is the key diagnostic for the RECALIBRATE vs REFIT decision:

- **GMCB (global MCB)** - the part of miscalibration fixed by multiplying all predictions by A/E. Cheap to fix.
- **LMCB (local MCB)** - the part requiring a model refit because the ranking is broken. Expensive.

If LMCB > GMCB, you need a refit - not just a scale correction.

In [ ]:
from insurance_monitoring import psi, csi, gini_coefficient

# Feature-level distribution checks
csi_table = csi(
    feat_ref,
    feat_cur,
    features=["driver_age", "vehicle_age", "ncd_years"],
)
print("CSI per feature:")
print(csi_table)
print()

# Model score PSI
score_psi = psi(
    reference=pred_ref,
    current=pred_cur,
    n_bins=10,
)
print(f"Score PSI: {score_psi:.4f}  ({'GREEN' if score_psi < 0.1 else 'AMBER' if score_psi < 0.25 else 'RED'})")
print()

# Gini change
g_ref = gini_coefficient(act_ref, pred_ref)
g_cur = gini_coefficient(act_cur, pred_cur)
print(f"Gini reference: {g_ref:.4f}  |  current: {g_cur:.4f}  |  change: {g_cur - g_ref:+.4f}")

## What you should see

| Check | Expected result | Why |
|-------|----------------|-----|
| `driver_age` CSI | AMBER or RED | Young drivers oversampled 2-3x |
| A/E ratio | ~1.08, AMBER | We inflated actuals by 8% |
| Gini change | Negative | 25% of predictions randomised |
| Recommendation | REFIT or INVESTIGATE | Multiple failure modes detected |

The aggregate A/E (~1.08) alone would read as borderline AMBER - not obviously alarming. The full report flags the distributional shift in driver age and the Gini decay, which together justify a refit rather than a simple recalibration.

At 3,000 monitoring policies the Gini drift z-test may not reach statistical significance - this is correct behaviour. The test is appropriately conservative at small sample sizes. At 15,000+ policies the same DGP produces z approximately -1.9, p approximately 0.06.

**GitHub:** https://github.com/burning-cost/insurance-monitoring  
**PyPI:** https://pypi.org/project/insurance-monitoring/  
**Blog:** [Your Pricing Model is Drifting](https://burning-cost.github.io/2026/03/07/your-pricing-model-is-drifting.html)